# 00 · Bajarse un modelo libre y mirarlo por dentro

**Demo rápida — Deep Learning para Business Analytics · Comillas ICADE**

Cinco técnicas, una sola historia. Trabajamos para **Banco Iberia**, un banco inventado que tiene
su normativa, sus clientes y su buzón de incidencias. Vamos a coger un modelo de lenguaje de pesos
abiertos y a hacerle las cinco cosas que se le hacen a un modelo en una empresa.

| | Cuaderno | Qué le hacemos al modelo | ¿Se tocan los pesos? |
|---|---|---|---|
| 1 | `01_rag` | le damos los documentos buenos en el prompt | no |
| 2 | `02_mcp` | le damos herramientas que ejecutan código | no |
| 3 | `03_sft` | le enseñamos una tarea con ejemplos resueltos | sí, todos |
| 4 | `04_lora` | lo mismo, moviendo el 3,5 % de los pesos | sí, unos pocos |
| 5 | `05_rlhf` | lo movemos hacia lo que las personas prefieren | sí, con un juez aprendido |

Este cuaderno es el punto de partida: descargar, abrir y entender qué es exactamente eso que
hemos descargado.

![El mapa de las cinco técnicas](figs/00_mapa.png)

La pregunta que ordena todo el curso de hoy es ésta: **¿el modelo falla porque no SABE algo, o
porque no HACE lo que quieres?**

Si no sabe algo (la comisión de tu banco, el inventario de hoy, el contrato que firmasteis ayer),
reentrenarlo es la respuesta cara y mala. Le das el dato: RAG o MCP.

Si sabe de sobra pero responde en un formato que no te sirve, con un tono que no es el de tu marca
o saltándose tu procedimiento, entonces sí hay que mover pesos: SFT, LoRA, RLHF.

![Quién usa esto](figs/00_quien_usa_que.png)

## 1. Qué necesitas instalado

En Google Colab, ejecuta la celda de abajo. En tu máquina, probablemente ya tengas casi todo.

**Requisitos de hardware**: todo está preparado para funcionar en CPU con unos **6 GB de RAM
libres**. Con GPU va mucho más rápido y no hay que cambiar nada: el código detecta el dispositivo
solo. Los tiempos que aparecen comentados en los cuadernos están medidos en un portátil sin GPU,
con 8 hilos.

In [ ]:
# En Colab, descomenta. En local, solo si te falta algo.
# !pip install -q transformers sentence-transformers peft accelerate faiss-cpu

In [ ]:
import sys
sys.path.insert(0, ".")          # para poder importar el paquete `comun`

from comun.config import (MODELO_CHARLA, MODELO_PEQUE, MODELO_EMBED, DEVICE,
                          DTYPE_CHARLA, DTYPE_ENTRENO, fijar_semilla, mil,
                          resumen_entorno, tam_mb, cronometro)

resumen_entorno()
fijar_semilla()

## 2. Los tres modelos de la demo

Los tres son de **pesos abiertos**: te los descargas, los ejecutas en tu máquina y nadie ve tus
datos. Esa es la diferencia operativa con llamar a una API, y en banca o sanidad es muchas veces
la diferencia entre poder hacer el proyecto y no poder.

| Modelo | Papel | Parámetros | Licencia | Dónde se usa |
|---|---|---|---|---|
| `Qwen/Qwen2.5-1.5B-Instruct` | conversa | 1.540 M | Apache 2.0 | cuadernos 01 y 02 |
| `HuggingFaceTB/SmolLM2-135M-Instruct` | se entrena | 135 M | Apache 2.0 | cuadernos 03, 04 y 05 |
| `intfloat/multilingual-e5-small` | vectoriza texto | 118 M | MIT | cuaderno 01 |

Apache 2.0 y MIT permiten uso comercial sin pedir permiso. No todas las licencias "abiertas" lo
hacen: la familia Llama, por ejemplo, va con licencia propia y con un límite de usuarios activos.
Antes de meter un modelo en un producto, el fichero que hay que leer es la licencia, no el
*leaderboard*.

In [ ]:
# Qué hay realmente dentro de un repositorio de Hugging Face.
from huggingface_hub import HfApi

api = HfApi()
info = api.model_info(MODELO_PEQUE, files_metadata=True)

print(f"Repositorio: {info.id}")
print(f"Licencia   : {info.cardData.get('license')}")
print(f"Descargas el último mes: {mil(info.downloads)}\n")
print(f"{'fichero':38s} {'tamaño':>10s}   para qué sirve")
print("-" * 86)
# El orden importa: se coge la PRIMERA clave que encaje, y
# "tokenizer_config.json" también acaba en "config.json".
PARA_QUE = {
    ".safetensors": "LOS PESOS. Es el modelo.",
    "tokenizer_config.json": "incluye el chat template (plantilla de conversación)",
    "generation_config.json": "valores por defecto al generar",
    "special_tokens_map.json": "tokens especiales: fin de texto, relleno...",
    "config.json": "arquitectura: capas, dimensiones, vocabulario",
    "tokenizer.json": "cómo se parte el texto en tokens",
    "vocab.json": "tabla token -> id",
    "merges.txt": "reglas de fusión del tokenizador BPE",
    "README.md": "la model card: licencia, datos de entrenamiento, limitaciones",
    "training_args.bin": "los hiperparámetros con los que lo entrenaron",
}
# Filtramos los subdirectorios: `onnx/` son conversiones a otro runtime y
# `runs/` son los logs de TensorBoard del entrenamiento original. Ninguno de
# los dos hace falta para usar el modelo desde PyTorch.
principales = [f for f in info.siblings if "/" not in f.rfilename]
for f in sorted(principales, key=lambda s: -(s.size or 0)):
    mb = (f.size or 0) / 1024**2
    clave = next((k for k in PARA_QUE if f.rfilename.endswith(k)), None)
    print(f"{f.rfilename:38s} {mb:9.1f} MB   {PARA_QUE.get(clave, '')}")

extra = [f for f in info.siblings if "/" in f.rfilename]
print(f"\n(+ {len(extra)} ficheros en subdirectorios: conversiones a ONNX y logs "
      f"del entrenamiento original, que no usamos)")

Fíjate en la proporción. El `.safetensors` es el 99 % del peso: eso son los números. Todo lo demás
son metadatos de unos pocos kilobytes que explican cómo usarlos.

**`safetensors` frente a `.bin`**: el formato antiguo de PyTorch (`.bin`) es un *pickle* de Python,
y un *pickle* puede ejecutar código arbitrario al abrirlo. `safetensors` es un formato de datos
puros que no puede ejecutar nada. Si te descargas pesos de un repositorio que no controlas, exige
`safetensors`.

In [ ]:
# Descargamos y abrimos el modelo pequeño. La primera vez tarda lo que tarde
# la red; a partir de ahí sale de ~/.cache/huggingface y es instantáneo.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

with cronometro("cargar SmolLM2-135M"):
    tok_p = AutoTokenizer.from_pretrained(MODELO_PEQUE)
    mod_p = AutoModelForCausalLM.from_pretrained(MODELO_PEQUE, dtype=DTYPE_ENTRENO)

cfg = mod_p.config
n_par = sum(p.numel() for p in mod_p.parameters())
print()
print(f"Parámetros            : {mil(n_par)}")
print(f"Capas (bloques)       : {cfg.num_hidden_layers}")
print(f"Dimensión del estado  : {cfg.hidden_size}")
print(f"Cabezas de atención   : {cfg.num_attention_heads}")
print(f"Vocabulario           : {mil(cfg.vocab_size)} tokens")
print(f"Contexto máximo       : {mil(cfg.max_position_embeddings)} tokens")
print(f"Ocupa en RAM          : {tam_mb(mod_p):.0f} MB en {DTYPE_ENTRENO}")

In [ ]:
# Un modelo es una pila de bloques iguales. Mira dónde está el peso.
from collections import defaultdict

por_grupo = defaultdict(int)
for nombre, p in mod_p.named_parameters():
    if "layers." in nombre:
        grupo = "bloques transformer (× %d)" % cfg.num_hidden_layers
    elif "embed" in nombre:
        grupo = "tabla de embeddings"
    else:
        grupo = "otros (norma final, cabeza)"
    por_grupo[grupo] += p.numel()

for g, n in sorted(por_grupo.items(), key=lambda kv: -kv[1]):
    print(f"{g:32s} {mil(n):>14s}   {100 * n / n_par:5.1f} %")

print("\nLos nombres de los módulos de UN bloque (los que tocaremos con LoRA):")
for nombre, _ in mod_p.named_parameters():
    if nombre.startswith("model.layers.0."):
        print("   ", nombre)

## 3. El tokenizador: el modelo no ve letras

Un modelo de lenguaje no procesa caracteres ni palabras. Procesa **tokens**, trozos de texto
sacados de un vocabulario fijo. Entender esto explica media docena de comportamientos raros: por
qué contar letras se le da mal, por qué el español cuesta más tokens que el inglés y por qué la
factura de una API sube cuando escribes en catalán.

In [ ]:
frase = "Banco Iberia concede hipotecas de hasta el 80 % del valor de tasación."

ids = tok_p(frase).input_ids
piezas = [tok_p.decode([i]) for i in ids]

print(f"Caracteres : {len(frase)}")
print(f"Tokens     : {len(ids)}")
print(f"Ratio      : {len(frase) / len(ids):.2f} caracteres por token\n")
print(" | ".join(repr(p)[1:-1] for p in piezas))

In [ ]:
# El mismo contenido en tres idiomas. El vocabulario de casi todos los modelos
# está dominado por el inglés, y eso se paga en tokens, en dinero y en contexto.
textos = {
    "inglés":    "The bank grants mortgages of up to 80 % of the appraised value.",
    "español":   "El banco concede hipotecas de hasta el 80 % del valor de tasación.",
    "euskera":   "Bankuak tasazio-balioaren % 80ra arteko hipotekak ematen ditu.",
}
for idioma, t in textos.items():
    n = len(tok_p(t).input_ids)
    print(f"{idioma:10s} {n:3d} tokens para {len(t):3d} caracteres "
          f"({len(t) / n:.2f} car/token)")

## 4. El chat template: lo que el modelo recibe de verdad

Cuando escribes en ChatGPT, lo que llega al modelo no es tu frase suelta. Es un texto plano con
marcas que separan los turnos. Esas marcas se decidieron durante el entrenamiento y son distintas
en cada familia de modelos.

El `tokenizer` trae la plantilla incrustada. Úsala siempre: construir el prompt a mano y
equivocarte en una marca degrada el modelo sin dar ningún error.

In [ ]:
mensajes = [
    {"role": "system", "content": "Eres el asistente de Banco Iberia. Responde breve."},
    {"role": "user", "content": "¿Qué es una hipoteca a tipo variable?"},
]

crudo = tok_p.apply_chat_template(mensajes, add_generation_prompt=True, tokenize=False)
print("--- TEXTO QUE RECIBE EL MODELO (SmolLM2) ---")
print(crudo)
print("--- fin ---")
print(f"\n`add_generation_prompt=True` es lo que añade la última línea: le dice al")
print("modelo que es su turno. Sin eso, el modelo sigue escribiendo tu mensaje.")

## 5. Generar texto

Generar es repetir un paso muy simple: el modelo da una distribución de probabilidad sobre los
~49.000 tokens del vocabulario, se elige uno, se añade al final y se vuelve a empezar.

Cómo se elige ese token es lo único que cambia entre los modos de generación:

- **greedy** (`do_sample=False`): siempre el más probable. Determinista. Sirve para tareas con una
  respuesta correcta, como clasificar.
- **muestreo con temperatura** (`do_sample=True, temperature=T`): se sortea. Con `T` baja se
  parece al greedy; con `T` alta se vuelve creativo y luego incoherente.

In [ ]:
# Miremos la distribución del primer token antes de elegir nada.
import torch

entrada = tok_p(crudo, return_tensors="pt", add_special_tokens=False)
with torch.no_grad():
    logits = mod_p(**entrada).logits[0, -1]       # logits del SIGUIENTE token

probs = torch.softmax(logits, dim=-1)
top = torch.topk(probs, 10)

print("Los 10 tokens más probables como primera palabra de la respuesta:\n")
for p, i in zip(top.values, top.indices):
    barra = "█" * int(p.item() * 60)
    print(f"  {repr(tok_p.decode([i]))[1:-1]:<16s} {p.item():6.2%}  {barra}")

In [ ]:
# Ahora el modelo que de verdad habla español. 1,5 B en bfloat16 ocupa ~3 GB.
from transformers import AutoModelForCausalLM, AutoTokenizer

with cronometro("cargar Qwen2.5-1.5B"):
    tok_c = AutoTokenizer.from_pretrained(MODELO_CHARLA)
    mod_c = AutoModelForCausalLM.from_pretrained(
        MODELO_CHARLA, dtype=DTYPE_CHARLA, low_cpu_mem_usage=True)

print(f"\nParámetros : {mil(sum(p.numel() for p in mod_c.parameters()))}")
print(f"Ocupa      : {tam_mb(mod_c):.0f} MB en {DTYPE_CHARLA}")
print(f"             {tam_mb(mod_c) * 2:.0f} MB si lo cargásemos en float32")

In [ ]:
def charlar(pregunta, sistema="Eres el asistente de Banco Iberia. Responde en dos frases.",
            max_new_tokens=90, temperatura=None):
    """Genera una respuesta con Qwen. `temperatura=None` significa greedy."""
    msgs = [{"role": "system", "content": sistema},
            {"role": "user", "content": pregunta}]
    texto = tok_c.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    ent = tok_c(texto, return_tensors="pt", add_special_tokens=False).to(mod_c.device)
    kw = dict(do_sample=False) if temperatura is None else dict(
        do_sample=True, temperature=temperatura, top_p=0.95)
    with torch.no_grad():
        salida = mod_c.generate(**ent, max_new_tokens=max_new_tokens,
                                pad_token_id=tok_c.eos_token_id, **kw)
    nuevos = salida[0, ent.input_ids.shape[1]:]
    return tok_c.decode(nuevos, skip_special_tokens=True).strip()


with cronometro("generar"):
    print(charlar("¿Qué es una hipoteca a tipo variable?"))

In [ ]:
# El efecto de la temperatura, con la misma pregunta tres veces.
fijar_semilla()
pregunta = "Escribe un eslogan para la nueva cuenta nómina de Banco Iberia."

for T in [None, 0.7, 1.5]:
    etiqueta = "greedy" if T is None else f"T = {T}"
    print(f"\n[{etiqueta}]")
    print("  ", charlar(pregunta, max_new_tokens=40, temperatura=T).replace("\n", " "))

## 6. El límite: lo que el modelo no puede saber

Banco Iberia está inventado. Sus documentos no existen en internet, así que ningún modelo los ha
visto durante el preentrenamiento. Preguntémosle de todas formas.

In [ ]:
print(charlar("¿Cuál es la comisión exacta por sacar dinero en un cajero de otro "
              "banco español según la normativa de Banco Iberia?",
              max_new_tokens=110))

Léelo con atención. Puede que haya dicho que no lo sabe, y eso sería lo correcto. Pero lo normal
es que haya dado una cifra con tono seguro, o una explicación genérica que suena a respuesta sin
serlo.

Esto no es un fallo que se arregle con un prompt mejor. El modelo está haciendo exactamente
aquello para lo que se le entrenó: continuar el texto de la manera más plausible. "Plausible" y
"cierto" coinciden casi siempre cuando el dato estaba en el preentrenamiento, y dejan de coincidir
en cuanto preguntas por algo que no estaba.

La respuesta real, que está en el `Catálogo de Productos Minoristas` de Banco Iberia, es **1,95 €
por operación**. Vamos a hacer que el modelo la lea en vez de inventarla: eso es el cuaderno 01.

## 7. Ejercicios

**0.1** Cambia `MODELO_CHARLA` en `comun/config.py` por `Qwen/Qwen2.5-0.5B-Instruct` y repite la
sección 5. ¿Cuánto se degrada el español al bajar de 1,5 B a 0,5 B? ¿Y cuánto gana en velocidad?

**0.2** La ratio caracteres/token del español sale peor que la del inglés. Calcula, para un
servicio que procese 10 millones de mensajes de clientes al mes a 0,15 € por millón de tokens de
entrada, cuánto cuesta esa diferencia al año.

**0.3** Sube la temperatura a 2.5 y genera veinte veces. Describe en qué momento exacto el texto
deja de ser español. Explica por qué pasa mirando lo que hace `softmax(logits / T)` cuando T crece.

**0.4** Usa `tok_p.apply_chat_template` y `tok_c.apply_chat_template` sobre los mismos mensajes y
compara los dos textos crudos. Son dos familias distintas de modelo. ¿Qué marcas usa cada una?

**0.5** Monta el prompt a mano, sin la plantilla, escribiendo tú `<|im_start|>user ... ` con una
marca mal escrita. Genera y describe cómo se degrada la respuesta. Este es el error silencioso más
caro que se comete al servir un modelo propio.